In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset

df = pd.read_csv('../data/campus_placement_data.csv')

print("Data successfully loaded! Total Rows:", len(df))
df.head(3)

In [ ]:
# Cleaning data by removing leakage and identifiers
df_clean = df.drop(columns=['salary_lpa', 'student_id'])

# Encoding variables
df_encoded = pd.get_dummies(df_clean, drop_first=True)

print("Data cleaning completed.")
print("New shape of the dataset:", df_encoded.shape)

In [ ]:
# Checking and handling missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])

# Filling missing values with the mode
df_cleaned = df_encoded.fillna(df_encoded.mode().iloc[0])
print("\nMissing values after filling:", df_cleaned.isnull().sum().max())

In [ ]:
from sklearn.model_selection import train_test_split
X = df_cleaned.drop(columns=['placed'])
y = df_cleaned['placed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Training the model
rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

print("Model training completed successfully!")

In [ ]:
from sklearn.metrics import classification_report

# Predicting on test set
y_pred = rf_model.predict(X_test)

# Printing the report
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

# Plotting the confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import joblib
joblib.dump(rf_model, '../app/placement_model.pkl')
joblib.dump(list(X_train.columns), '../app/model_columns.pkl')
print("New model exported successfully!")